# 1.6 Practical Application: Point-in-Time Quantitative Signal Research

## Overview

Quantitative research requires more than implementing mathematical formulas. Reliable research software must maintain temporal consistency, enforce explicit data contracts, prevent information leakage, and generate reproducible analytical results.

A common failure in financial modeling occurs when historical simulations use information that would not have been available at the time of an investment decision.

This problem, known as look-ahead bias, can produce misleading performance estimates even when the underlying Python code is syntactically correct.

This notebook develops a point-in-time quantitative signal research pipeline using Python's standard library.

The system evaluates momentum-based trading signals, applies delayed execution assumptions, calculates transaction-adjusted returns, and verifies important numerical and temporal invariants.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Design a modular quantitative research workflow.
2. Represent financial observations using immutable data models.
3. Validate historical price observations and configuration parameters.
4. Implement chronologically consistent signal generation.
5. Separate signal formation, execution, and performance evaluation.
6. Prevent common forms of look-ahead bias.
7. Implement long and short trading signal logic.
8. Calculate trade-level gross and net returns.
9. Incorporate transaction costs and fixed execution fees.
10. Aggregate research results using mathematically consistent conventions.
11. Apply professional naming, formatting, and documentation standards.
12. Develop deterministic regression tests and numerical invariants.
13. Evaluate computational complexity and software design trade-offs.
14. Identify methodological limitations in backtesting research.
15. Build reusable Python components suitable for further quantitative analysis.


## 1.6.1 Research Architecture and Problem Formulation

### Research Objective

Develop a deterministic framework for evaluating whether historical momentum signals are associated with profitable subsequent price movements after accounting for simplified transaction costs.

The system must distinguish between:

1. Historical information available at signal formation.
2. Signal generation.
3. Hypothetical execution.
4. Future price observations used for evaluation.
5. Trade-level performance measurement.
6. Aggregation and reporting.

### Research Architecture

The analytical pipeline follows this sequence:

    Raw Price Observations
            |
            v
    Data Validation
            |
            v
    Chronological Signal Generation
            |
            v
    Delayed Execution Assumption
            |
            v
    Gross Return Calculation
            |
            v
    Transaction Cost Adjustment
            |
            v
    Trade-Level Results
            |
            v
    Research Performance Summary
            |
            v
    Numerical and Temporal Verification

### Trading Signal

A simple momentum signal is defined using historical closing prices.

For asset i at session t:

    Momentum(i,t) = Price(i,t) / Price(i,t-L) - 1

where L is the historical lookback measured in observed trading sessions.

### Signal Classification

A positive momentum value exceeding a predefined threshold generates a LONG signal.

A sufficiently negative momentum value generates a SHORT signal.

Otherwise, no trade is generated.

### Execution Assumption

A signal formed using the closing price at session t is not executed at that same closing price.

Instead, the hypothetical position enters at the closing price of session t+1 and exits at the closing price of session t+2.

This separation avoids using the signal-formation close as a simultaneously guaranteed execution price.

### Performance Evaluation

For each hypothetical trade:

    GrossReturn = Direction * (ExitPrice / EntryPrice - 1)

where:

    Direction = +1 for LONG
    Direction = -1 for SHORT

Transaction costs are deducted from gross P&L to obtain net P&L.

### Research Scope

The implementation is a signal-level historical evaluation framework.

It is not a complete portfolio backtesting engine because it does not model shared capital constraints, overlapping position financing, or a portfolio equity curve.

### Technical Constraints

- Python 3.10 or later.
- Standard library only.
- No NumPy or Pandas.
- No external market data.
- Explicit type annotations.
- Immutable data models.
- Deterministic calculations.
- PEP 8-compatible formatting.
- No mutation of original price observations.


## 1.6.2 Financial Data Models and API Contracts

In [2]:
# ============================================================
# POINT-IN-TIME QUANTITATIVE SIGNAL RESEARCH
# ============================================================

from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from datetime import date
from math import fsum, isfinite
from typing import Literal


Direction = Literal["LONG", "SHORT"]

BPS_PER_UNIT = 10_000.0


def _finite_number(value: object) -> bool:
    """Return whether an object is a finite real number."""
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and isfinite(value)
    )


@dataclass(frozen=True, slots=True)
class PriceBar:
    """
    Represent a historical closing-price observation.

    Parameters
    ----------
    ticker : str
        Nonempty asset identifier.
    session : date
        Trading session associated with the observation.
    close_usd : float
        Strictly positive closing price in USD.
    """

    ticker: str
    session: date
    close_usd: float

    def __post_init__(self) -> None:
        """Validate the price observation."""

        if (
            not isinstance(self.ticker, str)
            or not self.ticker.strip()
        ):
            raise ValueError(
                "ticker must be a nonempty string."
            )

        if type(self.session) is not date:
            raise TypeError(
                "session must be a datetime.date."
            )

        if (
            not _finite_number(self.close_usd)
            or self.close_usd <= 0
        ):
            raise ValueError(
                "close_usd must be finite and positive."
            )


@dataclass(frozen=True, slots=True)
class ResearchConfig:
    """
    Store strategy and execution-cost assumptions.

    All return thresholds use decimal notation.
    """

    lookback_sessions: int
    threshold_decimal: float
    notional_usd: float
    round_trip_cost_bps: float
    fixed_fee_usd: float

    def __post_init__(self) -> None:
        """Validate research configuration."""

        if (
            isinstance(self.lookback_sessions, bool)
            or not isinstance(self.lookback_sessions, int)
            or self.lookback_sessions < 1
        ):
            raise ValueError(
                "lookback_sessions must be positive."
            )

        numeric_fields = {
            "threshold_decimal": self.threshold_decimal,
            "notional_usd": self.notional_usd,
            "round_trip_cost_bps": self.round_trip_cost_bps,
            "fixed_fee_usd": self.fixed_fee_usd,
        }

        if not all(
            _finite_number(value)
            for value in numeric_fields.values()
        ):
            raise ValueError(
                "Configuration values must be finite."
            )

        if not 0 < self.threshold_decimal < 1:
            raise ValueError(
                "threshold_decimal must be between 0 and 1."
            )

        if (
            self.notional_usd <= 0
            or self.round_trip_cost_bps < 0
            or self.fixed_fee_usd < 0
        ):
            raise ValueError(
                "Invalid notional or cost configuration."
            )


@dataclass(frozen=True, slots=True)
class SignalTrade:
    """Store one hypothetical signal evaluation."""

    ticker: str
    decision_session: date
    entry_session: date
    exit_session: date
    direction: Direction
    momentum_decimal: float
    entry_close_usd: float
    exit_close_usd: float
    gross_return_decimal: float
    gross_pnl_usd: float
    cost_usd: float
    net_pnl_usd: float


@dataclass(frozen=True, slots=True)
class ResearchReport:
    """Store aggregated signal research results."""

    trades: tuple[SignalTrade, ...]
    total_notional_usd: float
    total_gross_pnl_usd: float
    total_cost_usd: float
    total_net_pnl_usd: float
    mean_net_return_bps: float | None


print("Financial data models initialized.")


Financial data models initialized.


## 1.6.3 Historical Data Construction

### Data Requirements

Reliable historical analysis requires observations to follow a consistent temporal structure.

Each observation must contain:

- Asset identifier.
- Trading session.
- Closing price.
- Explicit currency convention.

### Chronological Ordering

Historical observations must be ordered by trading session.

The research framework must not silently interpret unordered prices as a valid chronological sequence.

### Duplicate Observations

Duplicate observations for the same asset and trading session should be rejected unless an explicit aggregation policy is defined.

### Missing Sessions

A missing trading session is not automatically equivalent to a missing price.

Financial markets do not trade on all calendar days.

This simplified framework measures lookback periods in observed trading sessions rather than calendar days.

### Synthetic Data

The example uses deterministic synthetic prices.

Synthetic observations allow calculations to be reproduced without external data dependencies.

They should not be interpreted as actual historical market prices.

### Point-in-Time Considerations

In a real financial dataset, additional information may be required to establish when an observation became available.

A session date alone is insufficient for modeling intraday publication delays or historical data revisions.

For the current application, closing prices are assumed to become available only after their respective sessions have completed.


In [3]:
# ============================================================
# SYNTHETIC HISTORICAL MARKET DATA
# ============================================================

sessions = tuple(
    date.fromisoformat(session)
    for session in (
        "2026-01-05",
        "2026-01-06",
        "2026-01-07",
        "2026-01-08",
        "2026-01-09",
        "2026-01-12",
        "2026-01-13",
        "2026-01-14",
        "2026-01-15",
    )
)

raw_prices = {
    "AAPL": [
        100, 101, 103, 106, 105,
        108, 110, 109, 112,
    ],
    "MSFT": [
        200, 201, 200, 198, 196,
        194, 197, 193, 192,
    ],
    "JPM": [
        150, 149, 151, 150, 153,
        155, 154, 157, 159,
    ],
}

price_history = {
    ticker: tuple(
        PriceBar(
            ticker=ticker,
            session=session,
            close_usd=float(close),
        )
        for session, close in zip(
            sessions,
            closes,
            strict=True,
        )
    )
    for ticker, closes in raw_prices.items()
}


# ============================================================
# RESEARCH CONFIGURATION
# ============================================================

config = ResearchConfig(
    lookback_sessions=3,
    threshold_decimal=0.015,
    notional_usd=10_000.0,
    round_trip_cost_bps=8.0,
    fixed_fee_usd=2.0,
)


print("RESEARCH DATA SUMMARY")
print("-" * 50)

for ticker, observations in price_history.items():
    print(
        f"{ticker:<8}"
        f"Observations: {len(observations):>3}  "
        f"Start: {observations[0].session}  "
        f"End: {observations[-1].session}"
    )


RESEARCH DATA SUMMARY
--------------------------------------------------
AAPL    Observations:   9  Start: 2026-01-05  End: 2026-01-15
MSFT    Observations:   9  Start: 2026-01-05  End: 2026-01-15
JPM     Observations:   9  Start: 2026-01-05  End: 2026-01-15


## 1.6.4 Historical Data Validation

### Objective

Validate historical price observations before performing any strategy calculations.

### Validation Requirements

**1. Nonempty Dataset**

The input must contain at least one asset.

**2. Sufficient Historical Observations**

A valid signal requires a historical lookback.

The evaluation also requires future observations for hypothetical entry and exit prices.

**3. Ticker Consistency**

Every observation must match the ticker assigned to its historical series.

**4. Strictly Increasing Sessions**

Observations must follow a strictly increasing chronological sequence.

**5. Valid Prices**

Closing prices must be finite and strictly positive.

**6. Immutable Inputs**

The validation process must not modify the original observations.

### Error Handling Policy

Invalid historical inputs should produce informative exceptions rather than silent corrections.

Unexpected automatic sorting or missing-value replacement may obscure underlying data-quality problems.

### Research Implications

A model can produce plausible financial results even when its input observations are incorrectly aligned.

Consequently, chronological validation is a methodological requirement rather than merely a programming convenience.


In [4]:

def validate_history(
    history: Mapping[str, Sequence[PriceBar]],
    config: ResearchConfig,
) -> None:
    """
    Validate chronological historical price observations.

    Parameters
    ----------
    history : Mapping[str, Sequence[PriceBar]]
        Asset-level historical price observations.
    config : ResearchConfig
        Validated research configuration.

    Raises
    ------
    ValueError
        If the dataset is empty, observations are
        insufficient, tickers are inconsistent,
        or sessions are not strictly increasing.
    """

    if not history:
        raise ValueError(
            "Historical dataset cannot be empty."
        )

    for ticker, bars in history.items():

        if (
            not isinstance(ticker, str)
            or not ticker.strip()
        ):
            raise ValueError(
                "History keys must be valid tickers."
            )

        # One lookback period, followed by an entry
        # session and an exit session.
        minimum_observations = (
            config.lookback_sessions
            + 3
        )

        if len(bars) < minimum_observations:
            raise ValueError(
                f"{ticker}: insufficient observations."
            )

        previous_session: date | None = None

        for bar in bars:

            if (
                not isinstance(bar, PriceBar)
                or bar.ticker != ticker
            ):
                raise ValueError(
                    f"{ticker}: invalid observation."
                )

            if (
                previous_session is not None
                and bar.session <= previous_session
            ):
                raise ValueError(
                    f"{ticker}: sessions must "
                    f"increase strictly."
                )

            previous_session = bar.session


validate_history(
    history=price_history,
    config=config,
)

print("Historical data validation passed.")


Historical data validation passed.


## 1.6.5 Point-in-Time Signal Generation

### Information Availability

At decision session t, a trading signal may only depend on information available at or before that decision point.

The framework assumes that the closing price at session t becomes available after that session has completed.

### Momentum Calculation

For a lookback of L observed sessions:

    Momentum_t = Close_t / Close_(t-L) - 1

### Signal Decision

    LONG  if Momentum_t >= Threshold

    SHORT if Momentum_t <= -Threshold

    FLAT  otherwise

A FLAT signal does not generate a trade.

### Temporal Separation

The research framework uses the following timeline:

    t-L          t          t+1          t+2
     |           |           |            |
     v           v           v            v
  Historical   Signal      Entry         Exit
   Reference   Decision     Price         Price

### Gross Return

For a LONG position:

    GrossReturn = ExitPrice / EntryPrice - 1

For a SHORT position:

    GrossReturn = -(ExitPrice / EntryPrice - 1)

### Simplified Transaction Costs

The model combines a proportional round-trip cost with a fixed round-trip fee.

    Cost = TradeNotional * CostBps / 10000
           + FixedFee

### Net P&L

    GrossPnL = TradeNotional * GrossReturn

    NetPnL = GrossPnL - Cost

### Important Methodological Distinction

Future prices are required to evaluate a historical signal.

However, future prices must not influence the signal that was originally generated.

The implementation must preserve this separation.

### Execution Limitation

The assumed next-session closing-price execution is hypothetical.

The implementation does not model whether a real trading order could obtain that exact closing price, nor does it simulate order types, market microstructure, or intraday execution uncertainty.


In [5]:

def generate_trades(
    history: Mapping[str, Sequence[PriceBar]],
    config: ResearchConfig,
) -> tuple[SignalTrade, ...]:
    """
    Generate and evaluate historical momentum signals.

    Signal information is restricted to observations
    available at or before the decision session.

    Hypothetical execution occurs at the next session's
    close, followed by liquidation one session later.

    Returns
    -------
    tuple[SignalTrade, ...]
        Chronologically consistent trade evaluations.
    """

    validate_history(history, config)

    trades: list[SignalTrade] = []

    cost_usd = (
        config.notional_usd
        * config.round_trip_cost_bps
        / BPS_PER_UNIT
        + config.fixed_fee_usd
    )

    for ticker, bars in sorted(history.items()):

        # The final two observations are reserved
        # for hypothetical entry and exit.
        for index in range(
            config.lookback_sessions,
            len(bars) - 2,
        ):

            current_close = bars[index].close_usd

            historical_close = bars[
                index - config.lookback_sessions
            ].close_usd

            # Signal formation uses historical
            # information available through index.
            momentum = (
                current_close / historical_close
                - 1
            )

            if momentum >= config.threshold_decimal:
                direction: Direction = "LONG"
                multiplier = 1

            elif momentum <= -config.threshold_decimal:
                direction = "SHORT"
                multiplier = -1

            else:
                continue

            # These future observations are used
            # only for retrospective trade evaluation.
            entry_bar = bars[index + 1]
            exit_bar = bars[index + 2]

            gross_return = (
                multiplier
                * (
                    exit_bar.close_usd
                    / entry_bar.close_usd
                    - 1
                )
            )

            gross_pnl = (
                config.notional_usd
                * gross_return
            )

            net_pnl = (
                gross_pnl
                - cost_usd
            )

            calculated_values = (
                momentum,
                gross_return,
                gross_pnl,
                net_pnl,
                cost_usd,
            )

            if not all(
                isfinite(value)
                for value in calculated_values
            ):
                raise OverflowError(
                    "Nonfinite financial calculation."
                )

            trades.append(
                SignalTrade(
                    ticker=ticker,
                    decision_session=bars[index].session,
                    entry_session=entry_bar.session,
                    exit_session=exit_bar.session,
                    direction=direction,
                    momentum_decimal=momentum,
                    entry_close_usd=entry_bar.close_usd,
                    exit_close_usd=exit_bar.close_usd,
                    gross_return_decimal=gross_return,
                    gross_pnl_usd=gross_pnl,
                    cost_usd=cost_usd,
                    net_pnl_usd=net_pnl,
                )
            )

    return tuple(trades)


trades = generate_trades(
    history=price_history,
    config=config,
)

print(f"Generated Trade Evaluations: {len(trades)}")


Generated Trade Evaluations: 9



## 1.6.6 Performance Aggregation and Reporting

### Aggregation Objective

Aggregate trade-level results without incorrectly interpreting them as a portfolio equity curve.

### Total Evaluated Notional

Each trade is evaluated using a fixed hypothetical notional.

    TotalEvaluatedNotional = NumberOfTrades * TradeNotional

This quantity measures the sum of notionals across evaluated trade instances.

It does not represent simultaneous invested capital or the required portfolio NAV.

### Total Gross P&L

    TotalGrossPnL = Sum(GrossPnL_i)

### Total Transaction Cost

    TotalCost = Sum(Cost_i)

### Total Net P&L

    TotalNetPnL = Sum(NetPnL_i)

### Mean Net Return in Basis Points

    MeanNetReturnBps =
        TotalNetPnL / TotalEvaluatedNotional * 10000

For equal trade notionals, this equals the arithmetic mean of individual net trade returns expressed in basis points.

### No-Trade Cases

When no trades are generated, total evaluated notional is zero.

In this case, a mean trade return is undefined.

The system should represent the result as missing rather than incorrectly reporting a realized return of zero.

### Portfolio-Level Limitation

Aggregate trade P&L is not equivalent to a fully financed, self-consistent portfolio backtest.

The framework does not account for simultaneous position funding, reinvestment, portfolio leverage, or capital allocation constraints.


In [6]:

def summarize_trades(
    trades: Sequence[SignalTrade],
    config: ResearchConfig,
) -> ResearchReport:
    """
    Aggregate hypothetical trade evaluation results.

    Parameters
    ----------
    trades : Sequence[SignalTrade]
        Evaluated historical trades.
    config : ResearchConfig
        Research configuration containing trade notional.

    Returns
    -------
    ResearchReport
        Aggregate P&L, evaluated notional, total costs,
        and mean trade return in basis points.

    Notes
    -----
    The mean trade return is not a compounded
    portfolio return.
    """

    total_notional = (
        len(trades)
        * config.notional_usd
    )

    total_gross_pnl = fsum(
        trade.gross_pnl_usd
        for trade in trades
    )

    total_cost = fsum(
        trade.cost_usd
        for trade in trades
    )

    total_net_pnl = fsum(
        trade.net_pnl_usd
        for trade in trades
    )

    if total_notional > 0:
        mean_net_return_bps = (
            BPS_PER_UNIT
            * total_net_pnl
            / total_notional
        )
    else:
        mean_net_return_bps = None

    aggregate_values = (
        total_notional,
        total_gross_pnl,
        total_cost,
        total_net_pnl,
    )

    if not all(
        isfinite(value)
        for value in aggregate_values
    ):
        raise OverflowError(
            "Nonfinite aggregate financial result."
        )

    return ResearchReport(
        trades=tuple(trades),
        total_notional_usd=total_notional,
        total_gross_pnl_usd=total_gross_pnl,
        total_cost_usd=total_cost,
        total_net_pnl_usd=total_net_pnl,
        mean_net_return_bps=mean_net_return_bps,
    )


report = summarize_trades(
    trades=trades,
    config=config,
)

print("Research performance aggregation completed.")


Research performance aggregation completed.


In [7]:

# ============================================================
# POINT-IN-TIME SIGNAL RESEARCH REPORT
# ============================================================

print("=" * 90)
print("POINT-IN-TIME QUANTITATIVE SIGNAL RESEARCH")
print("=" * 90)

print(
    f"{'Ticker':<9}"
    f"{'Decision':<14}"
    f"{'Direction':<11}"
    f"{'Momentum':>12}"
    f"{'Gross P&L':>15}"
    f"{'Net P&L':>15}"
)

print("-" * 90)

for trade in report.trades:

    print(
        f"{trade.ticker:<9}"
        f"{str(trade.decision_session):<14}"
        f"{trade.direction:<11}"
        f"{trade.momentum_decimal:>12.2%}"
        f"{trade.gross_pnl_usd:>15,.2f}"
        f"{trade.net_pnl_usd:>15,.2f}"
    )


print("\nAGGREGATE RESEARCH METRICS")
print("-" * 90)

print(f"Trade Evaluations: {len(report.trades)}")

print(
    f"Total Evaluated Notional: "
    f"${report.total_notional_usd:,.2f}"
)

print(
    f"Total Gross P&L: "
    f"${report.total_gross_pnl_usd:,.2f}"
)

print(
    f"Total Costs: "
    f"${report.total_cost_usd:,.2f}"
)

print(
    f"Total Net P&L: "
    f"${report.total_net_pnl_usd:,.2f}"
)

if report.mean_net_return_bps is not None:

    print(
        f"Mean Net Trade Return: "
        f"{report.mean_net_return_bps:.2f} bps"
    )

else:
    print("Mean Net Trade Return: N/A")


POINT-IN-TIME QUANTITATIVE SIGNAL RESEARCH
Ticker   Decision      Direction      Momentum      Gross P&L        Net P&L
------------------------------------------------------------------------------------------
AAPL     2026-01-08    LONG              6.00%         285.71         275.71
AAPL     2026-01-09    LONG              3.96%         185.19         175.19
AAPL     2026-01-12    LONG              4.85%         -90.91        -100.91
AAPL     2026-01-13    LONG              3.77%         275.23         265.23
JPM      2026-01-09    LONG              2.68%         -64.52         -74.52
JPM      2026-01-12    LONG              2.65%         194.81         184.81
JPM      2026-01-13    LONG              2.67%         127.39         117.39
MSFT     2026-01-09    SHORT            -2.49%        -154.64        -164.64
MSFT     2026-01-12    SHORT            -3.00%         203.05         193.05

AGGREGATE RESEARCH METRICS
--------------------------------------------------------------------


### Expected Results

#### Trade-Level Results

| Ticker | Decision Session | Direction | Net P&L |
|---|---|---|---:|
| AAPL | 2026-01-08 | LONG | +&#36;275.71 |
| AAPL | 2026-01-09 | LONG | +&#36;175.19 |
| AAPL | 2026-01-12 | LONG | -&#36;100.91 |
| AAPL | 2026-01-13 | LONG | +&#36;265.23 |
| JPM | 2026-01-09 | LONG | -&#36;74.52 |
| JPM | 2026-01-12 | LONG | +&#36;184.81 |
| JPM | 2026-01-13 | LONG | +&#36;117.39 |
| MSFT | 2026-01-09 | SHORT | -&#36;164.64 |
| MSFT | 2026-01-12 | SHORT | +&#36;193.05 |

#### Aggregate Metrics

| Metric | Expected Value |
|---|---:|
| Number of Evaluated Trades | 9 |
| Total Evaluated Notional | &#36;90,000.00 |
| Total Gross P&L | +&#36;961.30 |
| Total Transaction Costs | &#36;90.00 |
| Total Net P&L | +&#36;871.30 |
| Mean Net Trade Return | +96.81 bps |
| Positive Net P&L Trades | 6 |
| Negative Net P&L Trades | 3 |


### Interpretation

The strategy generates nine hypothetical trades across three assets.

The aggregate net P&L is positive under the specified synthetic data and execution assumptions.

However, the reported mean net trade return is not an annualized return, a Sharpe ratio, or a compounded portfolio return.

The sample is too small to support statistical conclusions about genuine trading profitability.

All observations are synthetic and should be interpreted as a programming and methodological demonstration.



## 1.6.7 Look-Ahead Bias and Temporal Verification

### Definition

Look-ahead bias occurs when a historical research process uses information that was unavailable at the time a prediction or decision was supposedly made.

### Common Sources

**1. Future Price Leakage**

Using later market observations when calculating earlier trading signals.

**2. Feature-Label Misalignment**

Allowing future outcome information to influence features or model parameters.

**3. Full-Sample Normalization**

Estimating statistics from the entire historical sample before evaluating earlier observations.

**4. Survivorship Bias**

Using only assets that remain in a dataset at the end of the historical period.

**5. Revised Data**

Using information that was revised after the historical decision date without reconstructing its original availability.

### Temporal Invariants

For every evaluated trade:

    DecisionSession < EntrySession < ExitSession

### Causal Stability

A correctly implemented historical signal should remain unchanged when price observations occurring after the decision are modified.

### Important Qualification

A signal may remain causally valid even though its realized P&L changes when future prices change.

This distinction is essential:

- Signal generation should depend only on available information.
- Historical performance evaluation necessarily uses later outcomes.

### Verification Strategy

Modify future price observations and verify that:

1. Earlier signal directions remain unchanged.
2. Historical momentum calculations remain unchanged.
3. Affected future P&L calculations change appropriately.
4. Earlier unaffected evaluations preserve their results.


In [8]:

# ============================================================
# LOOK-AHEAD BIAS DIAGNOSTIC
# ============================================================

original_trades = generate_trades(
    history=price_history,
    config=config,
)

# Construct a modified dataset without mutating
# the original historical observations.
modified_history = dict(price_history)

modified_history["AAPL"] = (
    *price_history["AAPL"][:-1],
    PriceBar(
        ticker="AAPL",
        session=sessions[-1],
        close_usd=500.0,
    ),
)

modified_trades = generate_trades(
    history=modified_history,
    config=config,
)


# ============================================================
# COMPARE SIGNAL DECISIONS
# ============================================================

def extract_signal_decisions(
    trades: Sequence[SignalTrade],
) -> tuple:
    """Extract decision information excluding future P&L."""

    return tuple(
        (
            trade.ticker,
            trade.decision_session,
            trade.direction,
            trade.momentum_decimal,
        )
        for trade in trades
    )


original_decisions = extract_signal_decisions(
    original_trades
)

modified_decisions = extract_signal_decisions(
    modified_trades
)

# Changing the last observed future price
# must not change previously formed signals.
assert original_decisions == modified_decisions


# ============================================================
# VERIFY TEMPORAL ORDERING
# ============================================================

for trade in original_trades:

    assert (
        trade.decision_session
        < trade.entry_session
        < trade.exit_session
    )


# ============================================================
# VERIFY FUTURE OUTCOME SENSITIVITY
# ============================================================

changed_trade_count = sum(
    original.net_pnl_usd != modified.net_pnl_usd
    for original, modified in zip(
        original_trades,
        modified_trades,
        strict=True,
    )
)

assert changed_trade_count == 1

print("Temporal ordering verified.")
print("Historical signal decisions remained unchanged.")
print("Future outcome sensitivity verified.")
print("Look-ahead diagnostic passed.")


Temporal ordering verified.
Historical signal decisions remained unchanged.
Future outcome sensitivity verified.
Look-ahead diagnostic passed.



## 1.6.8 Automated Testing and Numerical Invariants

### Testing Objective

Verify that the research pipeline produces mathematically consistent, deterministic, and chronologically valid results.

### Numerical Invariants

**1. Trade P&L Identity**

    NetPnL = GrossPnL - TransactionCost

**2. Gross P&L Aggregation**

    TotalGrossPnL = Sum(TradeGrossPnL)

**3. Transaction Cost Aggregation**

    TotalCost = Sum(TradeCost)

**4. Net P&L Aggregation**

    TotalNetPnL = Sum(TradeNetPnL)

**5. Temporal Ordering**

    DecisionSession < EntrySession < ExitSession

**6. No-Trade Behavior**

An empty trade collection should not generate a misleading realized return.

### Required Test Categories

- Deterministic reference calculations.
- Long and short signal behavior.
- Transaction-cost consistency.
- Temporal ordering.
- Invalid price observations.
- Duplicate or unordered sessions.
- Invalid research configurations.
- No-trade scenarios.
- Input immutability.
- Aggregate financial identities.

### Testing Principle

Financial research code requires both software-level tests and domain-specific numerical verification.

A syntactically correct implementation does not necessarily represent a valid trading strategy.


In [9]:

import unittest
from math import isclose


class SignalResearchTests(unittest.TestCase):
    """Validate the quantitative research pipeline."""

    def setUp(self) -> None:
        """Create a reproducible research environment."""
        self.history = price_history
        self.config = config

        self.trades = generate_trades(
            self.history,
            self.config,
        )

        self.report = summarize_trades(
            self.trades,
            self.config,
        )

    def test_reference_trade_count(self) -> None:
        """Verify the expected number of trades."""
        self.assertEqual(
            len(self.trades),
            9,
        )

    def test_reference_aggregate_pnl(self) -> None:
        """Verify independently calculated reference values."""
        self.assertAlmostEqual(
            self.report.total_gross_pnl_usd,
            961.3038486147851,
            places=6,
        )

        self.assertAlmostEqual(
            self.report.total_cost_usd,
            90.0,
        )

        self.assertAlmostEqual(
            self.report.total_net_pnl_usd,
            871.3038486147851,
            places=6,
        )

    def test_trade_accounting_identity(self) -> None:
        """Verify P&L calculations for every trade."""
        for trade in self.trades:

            self.assertTrue(
                isclose(
                    trade.net_pnl_usd,
                    trade.gross_pnl_usd
                    - trade.cost_usd,
                    rel_tol=0.0,
                    abs_tol=1e-9,
                )
            )

    def test_temporal_ordering(self) -> None:
        """Verify chronological decision and execution."""
        for trade in self.trades:

            self.assertLess(
                trade.decision_session,
                trade.entry_session,
            )

            self.assertLess(
                trade.entry_session,
                trade.exit_session,
            )

    def test_long_and_short_signals(self) -> None:
        """Verify support for both trade directions."""
        directions = {
            trade.direction
            for trade in self.trades
        }

        self.assertEqual(
            directions,
            {"LONG", "SHORT"},
        )

    def test_invalid_price(self) -> None:
        """Reject nonfinite market observations."""
        with self.assertRaises(ValueError):
            PriceBar(
                ticker="ABC",
                session=sessions[0],
                close_usd=float("nan"),
            )

    def test_invalid_configuration(self) -> None:
        """Reject negative research costs."""
        with self.assertRaises(ValueError):
            ResearchConfig(
                lookback_sessions=3,
                threshold_decimal=0.015,
                notional_usd=10_000.0,
                round_trip_cost_bps=-5.0,
                fixed_fee_usd=2.0,
            )

    def test_duplicate_session(self) -> None:
        """Reject non-increasing historical sessions."""
        invalid_history = dict(self.history)

        original = self.history["AAPL"]

        invalid_history["AAPL"] = (
            *original[:-1],
            PriceBar(
                ticker="AAPL",
                session=original[-2].session,
                close_usd=112.0,
            ),
        )

        with self.assertRaises(ValueError):
            validate_history(
                invalid_history,
                self.config,
            )

    def test_no_trade_behavior(self) -> None:
        """Do not invent a return when no trade exists."""
        restrictive_config = ResearchConfig(
            lookback_sessions=3,
            threshold_decimal=0.99,
            notional_usd=10_000.0,
            round_trip_cost_bps=8.0,
            fixed_fee_usd=2.0,
        )

        empty_trades = generate_trades(
            self.history,
            restrictive_config,
        )

        empty_report = summarize_trades(
            empty_trades,
            restrictive_config,
        )

        self.assertEqual(
            len(empty_trades),
            0,
        )

        self.assertIsNone(
            empty_report.mean_net_return_bps,
        )

    def test_input_preservation(self) -> None:
        """Verify that the original inputs remain unchanged."""
        original_history = {
            ticker: tuple(bars)
            for ticker, bars in self.history.items()
        }

        generate_trades(
            self.history,
            self.config,
        )

        self.assertEqual(
            original_history,
            self.history,
        )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    SignalResearchTests
)

test_result = unittest.TextTestRunner(
    verbosity=2
).run(suite)

assert test_result.wasSuccessful()


test_duplicate_session (__main__.SignalResearchTests.test_duplicate_session)
Reject non-increasing historical sessions. ... ok
test_input_preservation (__main__.SignalResearchTests.test_input_preservation)
Verify that the original inputs remain unchanged. ... ok
test_invalid_configuration (__main__.SignalResearchTests.test_invalid_configuration)
Reject negative research costs. ... ok
test_invalid_price (__main__.SignalResearchTests.test_invalid_price)
Reject nonfinite market observations. ... ok
test_long_and_short_signals (__main__.SignalResearchTests.test_long_and_short_signals)
Verify support for both trade directions. ... ok
test_no_trade_behavior (__main__.SignalResearchTests.test_no_trade_behavior)
Do not invent a return when no trade exists. ... ok
test_reference_aggregate_pnl (__main__.SignalResearchTests.test_reference_aggregate_pnl)
Verify independently calculated reference values. ... ok
test_reference_trade_count (__main__.SignalResearchTests.test_reference_trade_count)
Ver


## 1.6.9 Computational Complexity and Software Engineering

### Computational Complexity

Let:

- A denote the number of assets.
- T denote the average number of observations per asset.
- N denote the total number of historical observations.
- K denote the number of generated trades.

### Data Validation

Each observation is inspected once.

Time Complexity:

    O(N)

Auxiliary Space Complexity:

    O(1)

This excludes the input data already supplied to the function.

### Signal Generation

Each eligible observation is evaluated using a constant number of arithmetic operations.

The asset identifiers are also sorted for deterministic output ordering.

Time Complexity:

    O(N + A log A)

Output Space Complexity:

    O(K)

### Performance Aggregation

Each generated trade is processed once.

Time Complexity:

    O(K)

Space Complexity:

    O(K)

The reported trade sequence is stored as an immutable tuple.

### Engineering Considerations

**1. Deterministic Ordering**

Sorting asset identifiers ensures reproducible ordering of results.

**2. Immutable Records**

Frozen dataclasses help prevent accidental modification of observations and trade results.

**3. Explicit Assumptions**

The research configuration makes thresholds and execution costs visible.

**4. Separation of Responsibilities**

Validation, signal generation, aggregation, reporting, and testing are implemented as distinct operations.

**5. Reproducibility**

The same inputs and configuration produce the same outputs.

**6. Testability**

Independent components can be evaluated using deterministic test cases.

### Potential Improvements

- Memory-efficient streaming evaluation.
- Explicit missing-session policies.
- Independent signal and execution modules.
- Historical data provenance tracking.
- Structured experiment metadata.
- Automated parameter evaluation.
- Logging and observability.
- Integration with numerical computing libraries.

### Important Limitation

Improving computational performance does not establish research validity.

A faster backtesting implementation can still contain statistical bias or unrealistic execution assumptions.



## 1.6.10 Analytical Interpretation

### 1. Signal Formation

The research pipeline generates signals using historical momentum.

All signal decisions depend on observed prices available at or before the corresponding decision session.

This design enforces the intended temporal ordering between historical information and future outcomes.

### 2. Delayed Execution

Hypothetical execution occurs after the signal-formation session.

This prevents the model from assuming that it can simultaneously observe a closing price, generate a signal from that price, and execute at the same closing price.

Nevertheless, exact execution at the following session's close remains a simplifying assumption.

### 3. Trading Direction

The model supports both long and short directional signals.

Long trades benefit from positive subsequent price changes.

Short trades benefit from negative subsequent price changes.

Short financing costs, borrow availability, and margin requirements are not modeled.

### 4. Transaction Cost Adjustment

Trade-level gross P&L is reduced by proportional round-trip execution costs and fixed fees.

The example demonstrates how transaction costs can materially affect the profitability of individual signals.

### 5. Historical Research Results

The synthetic dataset produces nine evaluated trades and positive aggregate net P&L under the specified assumptions.

These results do not demonstrate statistical significance or predictive ability in real markets.

### 6. Temporal Integrity

The explicit separation between signal formation, entry, and exit reduces the risk of direct future-price leakage.

The temporal verification test confirms that modifying a later price observation does not change previously generated signal decisions in the tested scenario.

### 7. Aggregation Interpretation

Total evaluated trade notional is not equivalent to portfolio net 
asset value.

The mean net trade return should not be confused with a compounded portfolio return or an annualized performance measure.

### 8. Software Reliability

Immutable data models, explicit validation, type annotations, and automated tests improve the reliability and maintainability of the analytical implementation.

These engineering practices support reproducibility but do not eliminate financial modeling risk.

### 9. Statistical Limitations

The research framework does not perform:

- Statistical significance testing.
- Out-of-sample validation.
- Walk-forward parameter selection.
- Multiple-testing correction.
- Benchmark comparison.
- Risk-adjusted performance estimation.
- Portfolio-level capital accounting.
- Realistic execution simulation.

### 10. Research Methodology

A complete quantitative research process must distinguish between:

1. Strategy hypothesis formation.
2. Historical signal construction.
3. Parameter specification.
4. Historical simulation.
5. Statistical validation.
6. Out-of-sample evaluation.
7. Execution feasibility.
8. Economic interpretation.

### Final Assessment

The primary contribution of this notebook is the development of a chronologically consistent, reproducible, and testable Python research pipeline.

The implementation demonstrates how language fundamentals and professional coding conventions can be combined to support rigorous quantitative analysis.

The next stage of development should focus on statistical validation, realistic execution assumptions, portfolio-level accounting, and independent out-of-sample evaluation.



## Advanced Practice Exercises

### Exercise 1: Multi-Asset Signal Validation Framework

**Objective**

Extend the historical research pipeline to validate signals across multiple financial assets with different trading histories.

**Tasks**

1. Support assets with different numbers of observations.
2. Validate chronological ordering independently for each asset.
3. Identify missing and duplicate observations.
4. Define a policy for incomplete historical windows.
5. Distinguish insufficient history from valid FLAT signals.
6. Generate deterministic signal decisions.
7. Report data-quality violations before signal evaluation.
8. Verify that invalid observations cannot silently influence trading decisions.

**Requirements**

- Python standard library only.
- No mutation of original market data.
- Explicit error-handling policies.
- Comprehensive numerical and temporal tests.
- Professional API documentation.

---

### Exercise 2: Advanced Signal Classification Engine

**Objective**

Replace the single momentum threshold with a configurable multi-factor signal classification system.

**Tasks**

1. Implement short-term and medium-term momentum features.
2. Calculate trailing volatility using historical observations.
3. Introduce volatility-adjusted signal thresholds.
4. Define LONG, SHORT, and FLAT classifications.
5. Resolve conflicting factor signals.
6. Introduce a minimum data-quality requirement.
7. Document the financial interpretation of every feature.
8. Verify that feature calculations use only permitted historical information.

**Requirements**

- Avoid future information leakage.
- Use configurable model parameters.
- Support multiple assets.
- Include deterministic reference calculations.
- Document all statistical assumptions.

---

### Exercise 3: Parameter Sensitivity and Overfitting Analysis

**Objective**

Evaluate how signal performance changes under different parameter choices.

**Tasks**

1. Define multiple historical lookback lengths.
2. Define multiple signal thresholds.
3. Evaluate each parameter combination.
4. Record trade counts and average net trade returns.
5. Identify parameter combinations producing no trades.
6. Generate a comparative performance report.
7. Investigate sensitivity to small parameter changes.
8. Explain the difference between in-sample parameter selection and out-of-sample evaluation.

**Requirements**

- Preserve chronological integrity.
- Avoid selecting parameters using future evaluation data.
- Separate parameter estimation from testing.
- Include numerical regression checks.
- Document limitations due to sample size.

---

### Exercise 4: Transaction Cost Sensitivity Analysis

**Objective**

Develop a configurable transaction-cost model and evaluate its effects on trading signal performance.

**Tasks**

1. Support fixed execution fees.
2. Support proportional execution costs in basis points.
3. Introduce direction-dependent cost assumptions.
4. Compare gross and net trade returns.
5. Calculate break-even transaction-cost thresholds.
6. Identify trades that become unprofitable after costs.
7. Evaluate multiple cost scenarios.
8. Generate a trade-level cost attribution report.

**Requirements**

- Use explicit financial units.
- Maintain consistent return conventions.
- Support LONG and SHORT trades.
- Validate all cost parameters.
- Ensure aggregate cost identities remain correct.

---

### Exercise 5: Temporal Leakage Detection System

**Objective**

Design an automated testing framework for detecting future-information leakage in quantitative research pipelines.

**Tasks**

1. Define temporal invariants for signals and trades.
2. Modify future observations while preserving historical data.
3. Verify that historical signals remain unchanged.
4. Introduce deliberately leaky feature calculations.
5. Develop tests that detect those violations.
6. Investigate leakage caused by full-sample normalization.
7. Investigate leakage caused by parameter selection.
8. Produce a structured temporal integrity report.

**Requirements**

- No execution of untrusted source code.
- Deterministic tests.
- Clear distinction between feature data and outcome labels.
- Explicit definition of information availability.
- Documentation of undetectable or unsupported leakage cases.

---

### Exercise 6: Research Pipeline Refactoring and Optimization

**Objective**

Refactor the quantitative signal research framework into a modular Python package.

**Tasks**

1. Separate financial data models into dedicated modules.
2. Separate validation from signal calculations.
3. Extract execution assumptions into an independent component.
4. Separate performance aggregation and reporting.
5. Introduce structured logging.
6. Improve the handling of invalid observations.
7. Add type annotations to public APIs.
8. Implement automated unit and regression tests.
9. Profile computational bottlenecks.
10. Compare the behavior of the original and refactored implementations.

**Requirements**

- Follow PEP 8 and PEP 257.
- Preserve reference financial calculations.
- Avoid unnecessary global mutable state.
- Maintain reproducibility.
- Document changes in computational complexity.
- Support automated code-quality verification.



## Advanced Challenge: Production-Oriented Point-in-Time Research Framework

### Objective

Design and implement a modular quantitative research framework supporting chronological financial data processing, signal generation, trade evaluation, transaction-cost modeling, and statistical performance analysis.

The framework should be structured for reproducible research and future integration with NumPy, Pandas, and machine learning libraries.

### System Architecture

**1. Data Layer**

Implement:

- Historical price data models.
- Data validation.
- Chronological integrity checks.
- Missing-observation policies.
- Data provenance metadata.
- Point-in-time information constraints.

**2. Signal Layer**

Implement:

- Momentum-based features.
- Multiple configurable signal rules.
- LONG, SHORT, and FLAT classifications.
- Signal timestamp tracking.
- Historical feature-window validation.
- Signal reproducibility tests.

**3. Execution Layer**

Implement:

- Configurable execution delay.
- Transaction-cost assumptions.
- Hypothetical entry and exit rules.
- Direction-adjusted gross returns.
- Trade-level net P&L.
- Execution assumption documentation.

**4. Research Evaluation Layer**

Implement:

- Aggregate trade statistics.
- Performance attribution by asset.
- Positive and negative trade counts.
- Average gross and net trade returns.
- Parameter sensitivity analysis.
- Historical evaluation summaries.

**5. Temporal Integrity Layer**

Implement:

- Decision-time information validation.
- Feature timestamp verification.
- Future-observation perturbation tests.
- Detection of selected look-ahead violations.
- Chronological execution assertions.

**6. Software Quality Layer**

Implement:

- Immutable data models.
- Explicit type annotations.
- Structured docstrings.
- Professional naming conventions.
- Robust exception handling.
- Automated testing.
- Static analysis.
- Reproducible configuration.

### Advanced Requirements

1. Use Python 3.10 or later.
2. Separate public APIs from implementation details.
3. Preserve chronological consistency.
4. Explicitly define every financial calculation.
5. Avoid future information leakage.
6. Support multiple assets and signal configurations.
7. Handle invalid data predictably.
8. Maintain deterministic research results.
9. Implement numerical invariants.
10. Document statistical and computational limitations.
11. Support automated regression testing.
12. Design a reusable project architecture.

### Expected Deliverables

1. Modular Python source code.
2. Research configuration system.
3. Historical data validation module.
4. Signal generation engine.
5. Execution evaluation engine.
6. Transaction-cost analytics.
7. Research performance report.
8. Temporal integrity test suite.
9. Numerical regression tests.
10. Code-quality configuration.
11. Technical methodology documentation.
12. GitHub-ready project structure.

### Evaluation Criteria

| Dimension | Weight |
|---|---:|
| Temporal correctness | 25% |
| Financial calculation accuracy | 20% |
| Software architecture | 15% |
| Testing and validation | 15% |
| Code quality | 10% |
| Reproducibility | 10% |
| Documentation | 5% |

### Final Goal

Develop a quantitative research framework whose results can be reproduced, whose assumptions can be inspected, and whose calculations can be independently verified.

The framework should distinguish between code correctness, statistical validity, and economic feasibility.
